In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('train_final.csv')

In [3]:
train_df.head()

,Unnamed: 0,user_id,ts,gate_id
0,0,18,2022-07-29 09:08:54,7
1,1,18,2022-07-29 09:09:54,9
2,2,18,2022-07-29 09:09:54,9
3,3,18,2022-07-29 09:10:06,5
4,4,18,2022-07-29 09:10:08,5


In [6]:
train_df.ts.describe()

count                   37518
unique                  34068
top       2022-10-20 16:31:33
freq                        4
Name: ts, dtype: object

### --- Feature Engineering ---

In [22]:
train_df.columns.unique()

Index(['Unnamed: 0', 'user_id', 'ts', 'gate_id'], dtype='object')

In [ ]:
train_df['ts'] = pd.to_datetime(train_df['ts'])
train_df = train_df.sort_values(['user_id', 'ts'])

# Базовые признаки
train_df['hour'] = train_df['ts'].dt.hour
train_df['day_of_week'] = train_df['ts'].dt.dayofweek
train_df['day_of_month'] = train_df['ts'].dt.day
train_df['month'] = train_df['ts'].dt.month
train_df['minute'] = train_df['ts'].dt.minute

# Создание последовательностей для каждого пользователя
features_list = []
for user_id in train_df['user_id'].unique():
    user_data = train_df[train_df['user_id'] == user_id].copy()
    
    for lag in range(1, 4):
        user_data[f'gate_lag_{lag}'] = user_data['gate_id'].shift(lag)
    
    user_data['time_diff'] = user_data['ts'].diff().dt.total_seconds()
    user_data['gate_mean_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).mean()
    user_data['gate_std_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).std()
    
    def mode_func(x):
        if len(x) > 0:
            counts = x.value_counts()
            return counts.index[0] if not counts.empty else np.nan
        return np.nan
    
    user_data['gate_mode_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).apply(mode_func, raw=False)
    
    user_data['time_of_day'] = pd.cut(user_data['hour'], 
                                      bins=[0, 6, 12, 18, 24], 
                                      labels=['night', 'morning', 'afternoon', 'evening'],
                                      include_lowest=True)
    
    features_list.append(user_data)

train_features = pd.concat(features_list).reset_index(drop=True)

In [ ]:
# Заполнение пропусков
for col in train_features.columns:
    if train_features[col].dtype in ['float64', 'int64']:
        train_features[col] = train_features[col].fillna(train_features[col].median())
    elif train_features[col].dtype == 'object':
        train_features[col] = train_features[col].fillna('unknown')

# Подготовка для обучения
categorical_features = ['time_of_day', 'day_of_week']
for col in categorical_features:
    le = LabelEncoder()
    train_features[col] = le.fit_transform(train_features[col].astype(str))

# Кодирование target (user_id)
le_user = LabelEncoder()
train_features['user_id_encoded'] = le_user.fit_transform(train_features['user_id'])

numerical_features = [col for col in train_features.columns 
                     if col not in ['user_id', 'user_id_encoded', 'ts'] + categorical_features]

scaler = StandardScaler()
train_features[numerical_features] = scaler.fit_transform(train_features[numerical_features])

# Разделение данных
X = train_features.drop(columns=['user_id', 'user_id_encoded', 'ts'])
y = train_features['user_id_encoded']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Model

In [ ]:
# Обучение модели
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Оценка модели
y_pred = model.predict(X_val)
y_pred_original = le_user.inverse_transform(y_pred)
y_val_original = le_user.inverse_transform(y_val)

print(f"Validation Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_val_original, y_pred_original, zero_division=0))

Validation Accuracy: 0.1754
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.03      0.05       250
           1       0.64      0.11      0.19       260
           2       0.10      0.88      0.19         8
           3       0.29      0.28      0.29       198
           4       0.00      0.00      0.00         0
           5       0.10      0.50      0.17         2
           6       0.68      0.06      0.11       403
           7       0.06      1.00      0.12        10
           8       0.10      0.83      0.18         6
           9       0.14      0.33      0.19       207
          10       0.04      0.67      0.08         3
          11       0.47      0.03      0.05       256
          12       0.72      0.06      0.11       391
          14       0.20      0.65      0.30       139
          15       0.41      0.03      0.05       351
          17       0.26      0.53      0.35       135
          18       0.49      0

In [ ]:
# 2. Загрузка и подготовка тестовых данных
test_df = pd.read_csv('test_final.csv')
user_words = test_df['user_word'].copy()

test_df['ts'] = pd.to_datetime(test_df['ts'])
test_df = test_df.sort_values(['user_word', 'ts'])

test_df['hour'] = test_df['ts'].dt.hour
test_df['day_of_week'] = test_df['ts'].dt.dayofweek
test_df['day_of_month'] = test_df['ts'].dt.day
test_df['month'] = test_df['ts'].dt.month
test_df['minute'] = test_df['ts'].dt.minute

features_list_test = []
for user_word in test_df['user_word'].unique():
    user_data = test_df[test_df['user_word'] == user_word].copy()
    
    for lag in range(1, 4):
        user_data[f'gate_lag_{lag}'] = user_data['gate_id'].shift(lag)
    
    user_data['time_diff'] = user_data['ts'].diff().dt.total_seconds()
    user_data['gate_mean_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).mean()
    user_data['gate_std_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).std()
    
    user_data['gate_mode_3'] = user_data['gate_id'].rolling(window=3, min_periods=1).apply(mode_func, raw=False)
    
    user_data['time_of_day'] = pd.cut(user_data['hour'], 
                                      bins=[0, 6, 12, 18, 24], 
                                      labels=['night', 'morning', 'afternoon', 'evenvening'],
                                      include_lowest=True)
    
    features_list_test.append(user_data)

test_features = pd.concat(features_list_test).reset_index(drop=True)

# Заполнение пропусков
for col in test_features.columns:
    if test_features[col].dtype in ['float64', 'int64']:
        test_features[col] = test_features[col].fillna(test_features[col].median())
    elif test_features[col].dtype == 'object':
        test_features[col] = test_features[col].fillna('unknown')

# Кодирование категориальных признаков
for col in categorical_features:
    if col in test_features.columns:
        le = LabelEncoder()
        test_features[col] = le.fit_transform(test_features[col].astype(str))

# Масштабирование числовых признаков
test_features[numerical_features] = scaler.transform(test_features[numerical_features])

In [ ]:
# Предсказание
X_test = test_features.drop(columns=['user_word', 'ts'])
predictions_encoded = model.predict(X_test)
predictions = le_user.inverse_transform(predictions_encoded)

# 3. Сопоставление user_word с user_id
mapping_df = pd.DataFrame({
    'user_word': test_features['user_word'],
    'predicted_user_id': predictions
})

prob_df = mapping_df.groupby(['user_word', 'predicted_user_id']).size().reset_index(name='count')
prob_df['probability'] = prob_df['count'] / prob_df.groupby('user_word')['count'].transform('sum')
prob_df = prob_df.sort_values(['user_word', 'probability'], ascending=[True, False])

final_mapping_dict = {}
used_user_ids = set()

for user_word in prob_df['user_word'].unique():
    available = prob_df[
        (prob_df['user_word'] == user_word) & 
        (~prob_df['predicted_user_id'].isin(used_user_ids))
    ]
    
    if not available.empty:
        selected = available.iloc[0]
        final_mapping_dict[user_word] = selected['predicted_user_id']
        used_user_ids.add(selected['predicted_user_id'])
    else:
        final_mapping_dict[user_word] = -999

final_mapping = pd.DataFrame(list(final_mapping_dict.items()), 
                             columns=['user_word', 'preds'])

# Сохранение результата
final_mapping.to_csv('answer_final.csv', index=False)
print("Результат сохранен в answer_final.csv")

Результат сохранен в answer_final.csv
